In [17]:
import os
import torch
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader, Subset
from torchvision.utils import make_grid, save_image
import torch.nn as nn
import torch.nn.init as init
import torch.nn.functional as F
import sys
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.init as init
import torchvision
import torchvision.transforms as transforms
from torchvision.models import vgg16
from torch.utils.data import DataLoader
from tqdm import tqdm
import copy
import json
import time
import os
import logging
from datetime import datetime

torch.cuda.empty_cache()

# ============================================================
# 0. Настройка логирования
# ============================================================

# Создаем директорию для логов если её нет
os.makedirs("logs", exist_ok=True)

# Генерируем имя файла лога с временной меткой
log_filename = f"logs/vgg_attack_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

# Создаем файловый handler
file_handler = logging.FileHandler(log_filename, encoding='utf-8')
file_handler.setLevel(logging.INFO)

# Создаем консольный handler
console_handler = logging.StreamHandler(sys.__stdout__)
console_handler.setLevel(logging.INFO)

# Настраиваем формат
formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
file_handler.setFormatter(formatter)
console_handler.setFormatter(formatter)

# Получаем корневой логгер и настраиваем его
logger = logging.getLogger()
logger.setLevel(logging.INFO)

# Очищаем существующие handlers
for handler in logger.handlers[:]:
    logger.removeHandler(handler)

# Добавляем наши handlers
logger.addHandler(file_handler)
logger.addHandler(console_handler)

# Создаем отдельный логгер для нашего скрипта
script_logger = logging.getLogger(__name__)

# Сохраняем оригинальные stdout/stderr
original_stdout = sys.stdout
original_stderr = sys.stderr

# Создаем класс для перенаправления stdout/stderr
class TeeLogger:
    def __init__(self, logger, level=logging.INFO):
        self.logger = logger
        self.level = level
        self.buffer = []
        
    def write(self, message):
        if message and message.strip():
            # Очищаем буфер перед записью
            if self.buffer:
                self.flush()
            # Записываем сообщение напрямую, избегая рекурсии
            if self.level == logging.INFO:
                self.logger.info(message.strip())
            else:
                self.logger.error(message.strip())
    
    def flush(self):
        if self.buffer:
            combined = ''.join(self.buffer)
            if combined.strip():
                if self.level == logging.INFO:
                    self.logger.info(combined.strip())
                else:
                    self.logger.error(combined.strip())
            self.buffer = []
    
    def isatty(self):
        return False

# Перенаправляем stdout и stderr
sys.stdout = TeeLogger(script_logger, logging.INFO)
sys.stderr = TeeLogger(script_logger, logging.ERROR)

# Записываем информацию о запуске
script_logger.info("=" * 60)
script_logger.info(f"Запуск атаки ASE-CW на CIFAR-10 (VGG16+Tropical)")
script_logger.info(f"Время запуска: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
script_logger.info("=" * 60)

# ============================================================
# 1. Модель (VGG16 + Tropical) — полностью из нашего последнего обучения
# ============================================================

class StackedNet(nn.Module):
    def __init__(self, base_layers, top_layers):
        super().__init__()
        self.base_layers = base_layers
        self.top_layers = top_layers

    def forward(self, x):
        x = self.base_layers(x)
        x = x.view(x.size(0), -1)
        x = self.top_layers(x)
        return x


class TropEmbeddingLayer(nn.Module):
    def __init__(self, input_dim, num_classes, distance_type='symmetric',
                 initializer_w=None, lam=0.0):
        super().__init__()
        self.input_dim = input_dim
        self.num_classes = num_classes
        self.lam = lam
        if initializer_w is None:
            initializer_w = lambda t: init.normal_(t, mean=0.0, std=6.0)
        self.w = nn.Parameter(initializer_w(torch.empty(num_classes, input_dim)))
        self.bias = nn.Parameter(torch.zeros(num_classes))
        if distance_type == 'symmetric':
            self.distance_fn = self.symmetric_distance
        elif distance_type == 'max':
            self.distance_fn = self.max_distance
        else:
            raise ValueError("distance_type must be 'symmetric' or 'max'")

    def forward(self, x):
        x = x.view(-1, 1, self.input_dim)
        x = x.repeat(1, self.num_classes, 1)
        return self.distance_fn(x, -self.w)

    def symmetric_distance(self, x, w):
        diff = x - w
        return torch.max(diff, dim=2)[0] - torch.min(diff, dim=2)[0] + self.bias

    def max_distance(self, x, w):
        return torch.max(x - w, dim=2)[0] + self.bias


class ChangeSignLayer(nn.Module):
    def __init__(self, temperature=0.05):
        super().__init__()
        self.temperature = temperature

    def forward(self, distances):
        return -distances / self.temperature


class TropicalTopLayers(nn.Module):
    def __init__(self, input_dim, num_classes, distance_type='symmetric', temperature=0.05):
        super().__init__()
        self.layers = nn.Sequential(
            TropEmbeddingLayer(input_dim=input_dim, num_classes=num_classes,
                               distance_type=distance_type),
            ChangeSignLayer(temperature=temperature)
        )

    def forward(self, x):
        return self.layers(x)


class VGG16FeatureExtractor(nn.Module):
    def __init__(self, out_features=256):
        super().__init__()
        vgg = torchvision.models.vgg16(weights=None)
        self.features = vgg.features
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.projection = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, out_features),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = self.projection(x)
        return x


def build_vgg16_tropical(num_classes=10, feature_dim=256, device="cuda"):
    base = VGG16FeatureExtractor(out_features=feature_dim)
    top = TropicalTopLayers(input_dim=feature_dim, num_classes=num_classes)
    return StackedNet(base, top).to(device)


# ============================================================
# 2. CIFAR-10 данные + нормализация
# ============================================================

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)
normalize = transforms.Normalize(mean=CIFAR_MEAN, std=CIFAR_STD)

transform = transforms.Compose([transforms.ToTensor()])   # только [0,1]

script_logger.info("Загрузка тестового датасета CIFAR-10...")
test_dataset = datasets.CIFAR10(root="data", train=False, download=True, transform=transform)

# Изменено: теперь 1000 изображений (по 100 на каждый класс)
IMAGES_PER_CLASS = 100  # 100 изображений на каждый класс
selected_indices = []
class_count = {i: 0 for i in range(10)}

script_logger.info(f"Выборка {IMAGES_PER_CLASS} изображений из каждого класса (всего {IMAGES_PER_CLASS * 10} изображений)...")
for idx in range(len(test_dataset)):
    _, label = test_dataset[idx]
    label = int(label)
    if class_count[label] < IMAGES_PER_CLASS:
        selected_indices.append(idx)
        class_count[label] += 1
    # Проверяем, что набрали все классы
    if all(count >= IMAGES_PER_CLASS for count in class_count.values()):
        break

# Выводим статистику по классам
script_logger.info("Статистика выборки по классам:")
for class_id in range(10):
    script_logger.info(f"  Класс {class_id}: {class_count[class_id]} изображений")

subset = Subset(test_dataset, selected_indices)
loader = DataLoader(subset, batch_size=len(selected_indices), shuffle=False)
images, labels = next(iter(loader))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
script_logger.info(f"Используется устройство: {device}")

# ============================================================
# 3. Загрузка модели VGG16 + Tropical
# ============================================================

model_path = "VGG16_Tropical_CIFAR10_best.pth"
script_logger.info(f"Загрузка модели из {model_path}...")
state = torch.load(model_path, map_location=device)

model = build_vgg16_tropical(
    num_classes=10,
    feature_dim=state["feature_dim"],
    device=device
)
model.load_state_dict(state["model_state_dict"])
model.eval()

# Фильтруем только верно классифицированные изображения
with torch.no_grad():
    x_norm = normalize(images.to(device))
    preds = model(x_norm).argmax(dim=1).cpu()
    correct_mask = (preds == labels)
    images = images[correct_mask]
    labels = labels[correct_mask]

script_logger.info(f"После фильтрации осталось {len(images)} верно классифицированных изображений")

# Если осталось меньше 1000, можно дополнить другими изображениями или продолжить с тем что есть
if len(images) < 1000:
    script_logger.warning(f"Внимание: осталось только {len(images)} верно классифицированных изображений из 1000")

# ============================================================
# 4. Вспомогательные функции атаки (адаптированные)
# ============================================================

@torch.no_grad()
def get_tropical_params(model):
    trop = model.top_layers.layers[0]      # TropEmbeddingLayer
    change = model.top_layers.layers[1]    # ChangeSignLayer
    return trop.w.detach(), trop.bias.detach(), change.temperature


def linearized_distance(z, w, bias, class_idx, i_max, i_min):
    return (z[..., i_max] - z[..., i_min] +
            w[class_idx, i_max] - w[class_idx, i_min] +
            bias[class_idx])


def linearized_logit(z, w, bias, class_idx, i_max, i_min, temperature):
    d = linearized_distance(z, w, bias, class_idx, i_max, i_min)
    return -d / temperature


def current_candidate_classes(logits, true_label, top_m=3):
    mask = torch.ones_like(logits, dtype=torch.bool)
    mask[true_label] = False
    other_logits = logits[mask]
    other_classes = torch.arange(logits.numel(), device=logits.device)[mask]
    _, idx = torch.topk(other_logits, k=min(top_m, other_logits.numel()))
    return other_classes[idx]


def candidate_active_indices(z, w, class_idx, kmax=3, kmin=3):
    uk = z + w[class_idx]
    amax = torch.topk(uk, k=min(kmax, uk.numel()), largest=True).indices
    amin = torch.topk(uk, k=min(kmin, uk.numel()), largest=False).indices
    return amax, amin


def build_active_configurations(z, logits, y, w, kmax=3, kmin=3, n_competitors=3):
    configs = []
    y = int(y)
    competitors = current_candidate_classes(logits, y, top_m=n_competitors)
    amax_y, amin_y = candidate_active_indices(z, w, y, kmax=kmax, kmin=kmin)

    for j in competitors.tolist():
        amax_j, amin_j = candidate_active_indices(z, w, j, kmax=kmax, kmin=kmin)
        for iy_max in amax_y.tolist():
            for iy_min in amin_y.tolist():
                if iy_max == iy_min: continue
                for ij_max in amax_j.tolist():
                    for ij_min in amin_j.tolist():
                        if ij_max == ij_min: continue
                        configs.append((j, iy_max, iy_min, ij_max, ij_min))
    return configs


def config_surrogate_loss(delta, x0, y, config, model, w, bias, temperature,
                          c=1.0, kappa=0.0):
    x = (x0 + delta).clamp(0.0, 1.0)
    x_norm = normalize(x)
    z = model.base_layers(x_norm).view(1, -1)[0]

    j, iy_max, iy_min, ij_max, ij_min = config
    logit_y = linearized_logit(z, w, bias, int(y), iy_max, iy_min, temperature)
    logit_j = linearized_logit(z, w, bias, int(j), ij_max, ij_min, temperature)

    f = torch.clamp(logit_y - logit_j + kappa, min=0.0)
    l2 = torch.sum(delta * delta)
    return l2 + c * f


def simplex_projection(v):
    """Проекция на симплекс с улучшенной численной стабильностью"""
    # Если вектор имеет размер 1, возвращаем единицу
    if v.numel() == 1:
        return torch.ones_like(v)
    
    # Добавляем небольшой шум для стабильности
    v = v.clone()
    v = v + torch.randn_like(v) * 1e-8
    
    # Сортируем в порядке убывания
    u, _ = torch.sort(v, descending=True)
    
    # Вычисляем кумулятивную сумму
    cssv = torch.cumsum(u, dim=0) - 1
    
    # Находим rho
    ind = torch.arange(1, v.numel() + 1, device=v.device, dtype=v.dtype)
    cond = u - cssv / ind > 0
    
    # Проверяем, есть ли хотя бы одно истинное значение
    if not torch.any(cond):
        # Если нет, возвращаем вектор из равных значений
        return torch.ones_like(v) / v.numel()
    
    # Находим последний индекс, где условие выполняется
    rho = torch.nonzero(cond, as_tuple=False)[-1, 0]
    theta = cssv[rho] / (rho + 1.0)
    
    # Проецируем
    result = torch.clamp(v - theta, min=0.0)
    
    # Нормализуем для гарантии суммы 1 (на случай численных ошибок)
    sum_result = result.sum()
    if sum_result > 0:
        result = result / sum_result
    
    return result


def min_norm_combination(grads, iters=50, lr=0.1):
    G = torch.stack(grads, dim=1)
    n = G.size(1)
    if n == 1:
        g = G[:, 0]
        norm = g.norm(p=2)
        return torch.zeros_like(g) if norm < 1e-12 else -g / norm
    
    # Добавляем регуляризацию для стабильности
    eps = 1e-8
    Gram = G.t() @ G + eps * torch.eye(n, device=G.device)
    
    alpha = torch.full((n,), 1.0 / n, device=G.device, dtype=G.dtype)
    
    for _ in range(iters):
        grad_alpha = 2.0 * (Gram @ alpha)
        alpha = alpha - lr * grad_alpha
        alpha = simplex_projection(alpha)
    
    g_mix = G @ alpha
    norm = g_mix.norm(p=2)
    
    if norm < 1e-12:
        return torch.zeros_like(g_mix)
    else:
        return -g_mix / norm


@torch.no_grad()
def is_success(model, x_adv, y):
    x_adv = x_adv.clamp(0, 1)
    x_norm = normalize(x_adv)
    pred = model(x_norm).argmax(dim=1)
    return bool(pred.item() != int(y))


@torch.no_grad()
def exact_boundary_trace(model, x0, x1, y, n_steps=40, overshoot=1.02):
    direction = x1 - x0
    low, high = 0.0, 1.0
    if not is_success(model, (x0 + high * direction).clamp(0, 1), y):
        high = 2.0
        for _ in range(10):
            cand = (x0 + high * direction).clamp(0, 1)
            if is_success(model, cand, y):
                break
            high *= 1.5
    for _ in range(n_steps):
        mid = 0.5 * (low + high)
        cand = (x0 + mid * direction).clamp(0, 1)
        if is_success(model, cand, y):
            high = mid
        else:
            low = mid
    return (x0 + overshoot * high * direction).clamp(0, 1)


# ============================================================
# 5. Основная функция атаки ASE-CW
# ============================================================

def ase_cw_single(model, x0, y, eta=0.12, steps=180, c=8.0, kappa=0.0,
                  kmax=3, kmin=3, n_competitors=3, restarts=3,
                  early_stop_patience=30):
    w, bias, temperature = get_tropical_params(model)
    best_adv = x0.clone().detach()
    best_l2 = float("inf")

    for restart in range(restarts):
        if restart == 0:
            delta = torch.zeros_like(x0, requires_grad=True)
        else:
            noise = 0.05 * torch.randn_like(x0)
            delta = noise.detach().clone().requires_grad_(True)

        no_improve = 0
        for step_idx in range(steps):
            x = (x0 + delta).clamp(0.0, 1.0)
            x_norm = normalize(x)
            z = model.base_layers(x_norm).view(1, -1)[0]
            logits = model.top_layers(z.unsqueeze(0))[0].detach()

            configs = build_active_configurations(
                z.detach(), logits, int(y), w,
                kmax=kmax, kmin=kmin, n_competitors=n_competitors
            )
            if len(configs) == 0:
                break

            flat_grads = []
            for config in configs:
                if delta.grad is not None:
                    delta.grad.zero_()
                loss_s = config_surrogate_loss(delta, x0, int(y), config, model,
                                               w, bias, temperature, c, kappa)
                grad_s = torch.autograd.grad(loss_s, delta, retain_graph=False)[0]
                flat_grads.append(grad_s.detach().reshape(-1))

            direction_flat = min_norm_combination(flat_grads, iters=60, lr=0.15)
            direction = direction_flat.view_as(delta)

            with torch.no_grad():
                delta_new = delta + eta * direction
                x_new = (x0 + delta_new).clamp(0.0, 1.0)
                delta_new = x_new - x0
                current_l2 = torch.norm(delta_new.view(-1), p=2).item()

                if is_success(model, x_new, y):
                    if current_l2 < best_l2:
                        best_l2 = current_l2
                        best_adv = exact_boundary_trace(model, x0, x_new, y)
                        no_improve = 0
                    else:
                        no_improve += 1
                else:
                    no_improve += 1

                delta = delta_new.detach().requires_grad_(True)

            if no_improve >= early_stop_patience:
                break

    return best_adv


# ============================================================
# 6. Запуск атаки на 1000 изображениях с замером времени
# ============================================================

script_logger.info("\n=== ASE-CW: Active-Set Ensemble CW Attack на CIFAR-10 (VGG16+Tropical) ===")
script_logger.info(f"Всего изображений для атаки: {len(images)}")

adv_images = images.clone().detach().to(device)
attack_params = {
    "eta": 0.12, "steps": 180, "c": 8.0, "kappa": 0.0,
    "kmax": 3, "kmin": 3, "n_competitors": 3,
    "restarts": 3, "early_stop_patience": 30,
}

os.makedirs("adv_samples_cifar_vgg", exist_ok=True)

# Список для хранения времени обработки каждого изображения
processing_times = []
successful_attacks = 0
l2_values = []

# Запускаем атаку для всех изображений
for i in range(len(images)):
    start_time = time.time()  # Начало замера времени
    
    x0 = images[i:i+1].to(device)
    y = labels[i].item()

    adv = ase_cw_single(model=model, x0=x0, y=y, **attack_params)
    adv_images[i:i+1] = adv.cpu()
    
    end_time = time.time()  # Конец замера времени
    process_time = end_time - start_time
    processing_times.append(process_time)

    # Получаем предсказанный класс для adversarial примера
    with torch.no_grad():
        adv_pred = model(normalize(adv)).argmax(dim=1).item()
    
    l2_val = torch.norm(adv - x0).item()
    l2_values.append(l2_val)
    
    # Считаем успешные атаки
    if adv_pred != y:
        successful_attacks += 1
    
    # Сохраняем пару изображений (оригинал и adversarial)
    pair = torch.cat([x0.cpu(), adv.cpu()], dim=3)
    save_image(pair, f"adv_samples_cifar_vgg/sample_{i:04d}_class_{y}_to_{adv_pred}_l2_{l2_val:.4f}.png")

    # Выводим прогресс каждые 10 изображений
    if (i + 1) % 10 == 0:
        script_logger.info(f"Прогресс: {i+1}/{len(images)} обработано, успешно: {successful_attacks}, текущий success rate: {(successful_attacks/(i+1))*100:.1f}%")
    
    script_logger.info(f"[{i:04d}] true={y} → adv_pred={adv_pred} "
          f"L2={l2_val:.4f}  time={process_time:.2f}s → сохранено")

adv_images = adv_images.cpu()
images = images.cpu()

# Сохраняем времена в файл
times_file = "vgg_processing_times_1000.json"
with open(times_file, 'w') as f:
    json.dump({
        "processing_times": processing_times,
        "l2_values": l2_values,
        "mean_time": sum(processing_times) / len(processing_times),
        "total_time": sum(processing_times),
        "num_images": len(processing_times),
        "successful_attacks": successful_attacks,
        "attack_success_rate": (successful_attacks / len(processing_times)) * 100
    }, f, indent=4)

mean_time = sum(processing_times) / len(processing_times)
success_rate = (successful_attacks / len(processing_times)) * 100

script_logger.info(f"\nСреднее время обработки одного изображения: {mean_time:.2f} секунд")
script_logger.info(f"Общее время обработки: {sum(processing_times):.2f} секунд")
script_logger.info(f"Времена сохранены в файл: {times_file}")

with torch.no_grad():
    x_norm = normalize(adv_images.to(device))
    preds_adv = model(x_norm).argmax(dim=1).cpu()
    success_mask = (preds_adv != labels)
    success_rate_verify = success_mask.float().mean().item() * 100
    l2_all = torch.norm((adv_images - images).view(len(images), -1), p=2, dim=1)
    mean_l2 = l2_all.mean().item()

script_logger.info(f"\nASE-CW — ИТОГ (CIFAR-10, {len(images)} изображений):")
script_logger.info(f"Success rate: {success_rate_verify:.1f}%")
script_logger.info(f"Mean L2: {mean_l2:.4f}")
script_logger.info(f"Успешно атаковано: {int(success_mask.sum())} из {len(images)}")

# Сохранение результатов
results = {
    "adv": adv_images,
    "orig": images,
    "labels": labels,
    "preds_adv": preds_adv,
    "attack_params": attack_params,
    "processing_times": processing_times,
    "l2_values": l2_values,
    "mean_time": mean_time,
    "total_time": sum(processing_times),
    "success_rate": success_rate_verify,
    "mean_l2": mean_l2,
    "successful_attacks": int(success_mask.sum()),
    "total_images": len(images)
}

torch.save(results, "ase_cw_vgg16_cifar10_1000.pt")

# Визуализация (показываем первые 64 изображения)
num_show = min(64, len(images))
clean_grid = make_grid(images[:num_show], nrow=8, normalize=True, padding=2)
adv_grid = make_grid(adv_images[:num_show], nrow=8, normalize=True, padding=2)
combined = torch.cat([clean_grid, adv_grid], dim=2)
save_image(combined, "vgg_ase_cw_cifar_visual_comparison_1000.png")

script_logger.info("\nГотово!")
script_logger.info("Сохранено:")
script_logger.info(" • ase_cw_vgg16_cifar10_1000.pt")
script_logger.info(" • vgg_ase_cw_cifar_visual_comparison_1000.png")
script_logger.info(f" • {times_file}")
script_logger.info(" • все пары в папке adv_samples_cifar_vgg/ (1000 изображений)")
script_logger.info(f" • лог выполнения: {log_filename}")

# Сохраняем дополнительную информацию о запуске
summary_info = {
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "model_path": model_path,
    "model_type": "VGG16_Tropical",
    "device": str(device),
    "num_images": len(images),
    "attack_params": attack_params,
    "results": {
        "success_rate": success_rate_verify,
        "mean_l2": mean_l2,
        "successful_attacks": int(success_mask.sum()),
        "mean_time_per_image": mean_time,
        "total_time": sum(processing_times)
    },
    "log_file": log_filename
}

with open("vgg_attack_summary_1000.json", 'w') as f:
    json.dump(summary_info, f, indent=4)

script_logger.info(" • vgg_attack_summary_1000.json")

# Восстанавливаем stdout и stderr
sys.stdout = original_stdout
sys.stderr = original_stderr

# Выводим финальное сообщение в консоль
print(f"\nАтака завершена! Лог сохранен в: {log_filename}")
print(f"Результаты сохранены в файлы:")
print(f"  • ase_cw_vgg16_cifar10_1000.pt")
print(f"  • vgg_ase_cw_cifar_visual_comparison_1000.png")
print(f"  • vgg_processing_times_1000.json")
print(f"  • vgg_attack_summary_1000.json")
print(f"  • adv_samples_cifar_vgg/ (папка с 1000 изображениями)")

script_logger.info("Выполнение завершено!")